<a href="https://colab.research.google.com/github/Theresa-Gao-0529/MSSP607/blob/main/Week_6_Lambda_Function.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Participation Activity Week 6: Lambda Function for a DataFrame



## 1. Import pandas and load the data



In [6]:
from pathlib import Path
import pandas as pd

DATA_FILE = Path("Student Performance Data.csv")

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if not DATA_FILE.exists():
    if IN_COLAB:
        uploaded = files.upload()
        if DATA_FILE.name not in uploaded:
            raise FileNotFoundError(f"Please upload {DATA_FILE.name}.")
    else:
        raise FileNotFoundError(
            f"Could not find {DATA_FILE.name}. Put it in the same folder as this notebook."
        )

students = pd.read_csv(DATA_FILE)
print(f"Dataset shape: {students.shape[0]} rows × {students.shape[1]} columns")
display(students.head())

Saving Student Performance Data.csv to Student Performance Data.csv
Dataset shape: 1000 rows × 10 columns


,gender,race/ethnicity,parental level of education,lunch,math score,reading score,writing score,total score,test preparation course,admission prospect
0,female,group B,bachelor's degree,standard,72,72,74,218,none,medium
1,female,group C,some college,standard,69,90,88,247,completed,high
2,female,group B,master's degree,standard,90,95,93,278,none,high
3,male,group A,associate's degree,free/reduced,47,57,44,148,none,low
4,male,group C,some college,standard,76,78,75,229,none,high


## 2. Define the score columns



In [7]:
score_columns = ["math score", "reading score", "writing score"]
required_columns = ["gender", "test preparation course", *score_columns]
missing_columns = [column for column in required_columns if column not in students.columns]
if missing_columns:
    raise KeyError(f"Missing required columns: {missing_columns}")

students[score_columns].describe()

,math score,reading score,writing score
count,1000.00000,1000.000000,1000.000000
mean,66.08900,69.169000,68.054000
std,15.16308,14.600192,15.195657
min,0.00000,17.000000,10.000000
25%,57.00000,59.000000,57.750000
50%,66.00000,70.000000,69.000000
75%,77.00000,79.000000,79.000000
max,100.00000,100.000000,100.000000


## 3. Create a new DataFrame with a lambda function



In [8]:
performance_summary = lambda df: (
    df.loc[:, ["gender", "test preparation course", *score_columns]]
    .copy()
    .assign(
        average_score=lambda summary: summary[score_columns].mean(axis=1).round(2)
    )
    .assign(
        performance_level=lambda summary: summary["average_score"].apply(
            lambda score: (
                "High" if score >= 80
                else "Satisfactory" if score >= 60
                else "Needs support"
            )
        )
    )
)

summary = performance_summary(students)
print(f"New DataFrame shape: {summary.shape[0]} rows × {summary.shape[1]} columns")
display(summary.head(10))

New DataFrame shape: 1000 rows × 7 columns


,gender,test preparation course,math score,reading score,writing score,average_score,performance_level
0,female,none,72,72,74,72.67,Satisfactory
1,female,completed,69,90,88,82.33,High
2,female,none,90,95,93,92.67,High
3,male,none,47,57,44,49.33,Needs support
4,male,none,76,78,75,76.33,Satisfactory
5,female,none,71,83,78,77.33,Satisfactory
6,female,completed,88,95,92,91.67,High
7,male,none,40,43,39,40.67,Needs support
8,male,completed,64,64,67,65.00,Satisfactory
9,female,none,38,60,50,49.33,Needs support


## 4. Review and save the result


In [9]:
display(summary["performance_level"].value_counts().rename_axis("performance_level").to_frame("student_count"))

OUTPUT_FILE = Path("Student Performance Summary.csv")
summary.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(summary)} rows to {OUTPUT_FILE}")

if IN_COLAB:
    files.download(str(OUTPUT_FILE))

,student_count
performance_level,
Satisfactory,517
Needs support,285
High,198


Saved 1000 rows to Student Performance Summary.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>